In [1]:
import pandas as pd

# Load train and test dataframe
train = pd.read_csv('/kaggle/input/math-10/train.csv')
test  = pd.read_csv('/kaggle/input/math-10/test.csv')


# Identify features and target
X = [
    'frequency','attack-angle','chord-length','free-stream-velocity', 'displacement-thickness'
]
y = 'scaled-sound-pressure'


# Create training X and y
X_train = train[X]
y_train = train[y]



import numpy as np
from xgboost import XGBRegressor
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score

# 🧠 Step 1: Feature Engineering on X_train
X_train["freq_vel_interaction"] = X_train["frequency"] * X_train["free-stream-velocity"]
X_train["strouhal"] = (X_train["frequency"] * X_train["chord-length"]) / X_train["free-stream-velocity"]
X_train["strouhal_squared"] = X_train["strouhal"] ** 2
X_train["strouhal_cubed"] = X_train["strouhal"] ** 3

X_train["strouhal_angle"] = X_train["strouhal"] * X_train["attack-angle"]

X_train["log_strouhal_angle"] = np.log1p(X_train["strouhal_angle"])



#X_train["log_frequency"] = np.log1p(X_train["frequency"])
#X_train["log_velocity"] = np.log1p(X_train["free-stream-velocity"])
#X_train["freq_squared"] = X_train["frequency"] ** 2
#X_train["velocity_squared"] = X_train["free-stream-velocity"] ** 2
X_train["velocity5_thickness"] = (X_train["free-stream-velocity"] ** 5) * X_train["displacement-thickness"]
#X_train["log_velocity5_thickness"] = np.log1p(X_train["velocity5_thickness"])
X_train["angle_vel_interaction"] = X_train["attack-angle"] * X_train["free-stream-velocity"]
X_train["angle_freq_interaction"] = X_train["attack-angle"] * X_train["frequency"]
X_train["angle_chord_interaction"] = X_train["attack-angle"] * X_train["chord-length"]

X_train["angle_squared"] = X_train["attack-angle"] ** 2
X_train["angle_freq_squared"] = X_train["attack-angle"] * (X_train["frequency"] ** 2)




X_train.drop(columns=["velocity5_thickness", "strouhal_angle", "strouhal"], inplace=True)



# ⚙️ Step 3: Define pipeline
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('xgb', XGBRegressor(objective='reg:squarederror', random_state=1))
])

param_grid = {
    'xgb__n_estimators': [480],
    'xgb__max_depth': [9], #out of [3,5,7]
    'xgb__learning_rate': [0.10704067150254538],

    'xgb__subsample': [0.8], #chose 0.8 and 0.66
    'xgb__colsample_bytree': [1.0], #0.5 to 1
    'xgb__reg_alpha': [0.1],
    'xgb__reg_lambda': [10], #out of [1,5,10]
    'xgb__min_child_weight': [5],
    'xgb__gamma': [0],
}

# 🔍 Step 5: GridSearchCV setup
grid = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    scoring='r2',
    cv=12,
    n_jobs=-1,
    verbose=1
)



# 🚀 Step 6: Fit model to training data
grid.fit(X_train, y_train)


# Now access best_index_ safely
best_index = grid.best_index_
mean_r2 = grid.cv_results_['mean_test_score'][best_index]
std_r2 = grid.cv_results_['std_test_score'][best_index]

#print("Best Mean CV R²:", mean_r2)
#print("Standard Deviation of CV R²:", std_r2)

# 📈 Step 7: Evaluation
#print("✅ Best Params:", grid.best_params_)
#print("📈 Best CV R²:", grid.best_score_)

# Training R² on full training data
best_model = grid.best_estimator_
train_r2 = best_model.score(X_train, y_train)
#print("🎯 Training R²:", train_r2)



X_test = test[X].copy()

# Apply same feature engineering to test set
X_test["freq_vel_interaction"] = X_test["frequency"] * X_test["free-stream-velocity"]
X_test["strouhal"] = (X_test["frequency"] * X_test["chord-length"]) / X_test["free-stream-velocity"]
X_test["strouhal_squared"] = X_test["strouhal"] ** 2
X_test["strouhal_cubed"] = X_test["strouhal"] ** 3

X_test["strouhal_angle"] = X_test["strouhal"] * X_test["attack-angle"]
X_test["log_strouhal_angle"] = np.log1p(X_test["strouhal_angle"])

# Uncomment below if log versions are needed
# X_test["log_frequency"] = np.log1p(X_test["frequency"])
# X_test["log_velocity"] = np.log1p(X_test["free-stream-velocity"])
# X_test["freq_squared"] = X_test["frequency"] ** 2
# X_test["velocity_squared"] = X_test["free-stream-velocity"] ** 2

X_test["velocity5_thickness"] = (X_test["free-stream-velocity"] ** 5) * X_test["displacement-thickness"]
#X_test["log_velocity5_thickness"] = np.log1p(X_test["velocity5_thickness"])

X_test["angle_vel_interaction"] = X_test["attack-angle"] * X_test["free-stream-velocity"]
X_test["angle_freq_interaction"] = X_test["attack-angle"] * X_test["frequency"]
X_test["angle_chord_interaction"] = X_test["attack-angle"] * X_test["chord-length"]

X_test["angle_squared"] = X_test["attack-angle"] ** 2
X_test["angle_freq_squared"] = X_test["attack-angle"] * (X_test["frequency"] ** 2)

# Optional: drop intermediate columns if not needed
X_test.drop(columns=["velocity5_thickness", "strouhal_angle","strouhal"], inplace=True)

# Now make predictions
preds = best_model.predict(X_test)


# Prepare submission DataFrame with column 'id' and target 'scaled-sound-pressure'
submission = pd.DataFrame({
    'id':   test['id'],
    'scaled-sound-pressure': preds
})

print(submission)

# Save submission file
submission.to_csv('submission.csv', index=False)


Fitting 12 folds for each of 1 candidates, totalling 12 fits
      id  scaled-sound-pressure
0      0             116.537659
1      1             134.222656
2      2             118.741714
3      3             129.583298
4      4             124.970833
..   ...                    ...
597  597             128.387756
598  598             130.255005
599  599             111.809692
600  600             119.125847
601  601             112.286583

[602 rows x 2 columns]
